# 74. The VRP with Backhauls Problem

## Tier 1: The Pen & Paper Method (Network Flow Formulation)

### Key assumptions
- Complete directed graph with known travel costs between all locations
- Vehicle capacity is uniform and limited
- All deliveries must be completed before any pickups (precedence constraint)
- Each customer is visited exactly once
- Vehicles start and end at the depot

### Approach (step-by-step)
1. **Problem Definition**: Define the VRPB on a complete directed graph with depot, linehaul (delivery), and backhaul (pickup) customers
2. **Mathematical Formulation**: Create a mixed-integer programming model with binary routing variables and continuous load variables
3. **Constraint Implementation**: Implement capacity constraints, precedence constraints, and flow conservation
4. **Solution Method**: Use exhaustive search for small instances to find optimal solutions
5. **Validation**: Verify that all constraints are satisfied in the optimal solution

### What to look for in the results
- Optimal route sequences respecting delivery-before-pickup precedence
- Vehicle load tracking throughout the routes
- Total travel distance minimization while satisfying all constraints
- Proper capacity utilization without exceeding vehicle limits

### Concrete example (from the source)
The source provides a small instance with:
- 4 delivery customers with demands: d₁=5, d₂=4, d₃=6, d₄=3
- 3 pickup customers with demands: p₅=4, p₆=5, p₇=3
- Vehicle capacity Q=20
- 8×8 symmetric distance matrix

Expected optimal solution: Route 0→1→2→3→4→5→6→7→0 with total distance 25

### Why this Tier exists vs earlier Tiers
This is the foundational Tier that provides:
- **Mathematical rigor**: Exact formulation with provable optimality
- **Baseline for comparison**: Reference point for evaluating heuristic methods
- **Constraint understanding**: Clear modeling of precedence and capacity constraints
- **Theoretical foundation**: Basis for understanding VRPB structure

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution for small instances
- Provides complete mathematical understanding of the problem
- Serves as validation benchmark for approximate methods
- Clear constraint modeling and verification

**Cons:**
- Computationally intractable for large instances (exponential complexity)
- Requires specialized optimization software for larger problems
- Limited to small problem sizes in practice
- May be overkill for quick decision-making scenarios

### When to use this Tier
- **Small instances**: When the problem size allows exact solution
- **Benchmarking**: To validate heuristic or metaheuristic approaches
- **Academic settings**: For teaching VRPB mathematical formulation
- **Critical applications**: When optimality is essential and problem size is manageable

In [ ]:
# Import required libraries for mathematical optimization and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import permutations, combinations
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully for VRPB Network Flow Formulation")

In [ ]:
@dataclass
class VRPBInstance:
    """Data class for VRP with Backhauls problem instance"""
    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray
    
    def __post_init__(self):
        self.num_customers = self.num_deliveries + self.num_pickups
        # Customer indices: 0=depot, 1..n=deliveries, n+1..n+m=pickups
        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(range(self.num_deliveries + 1, 
                                         self.num_deliveries + self.num_pickups + 1))
        self.all_customer_indices = self.delivery_indices + self.pickup_indices

@dataclass
class Route:
    """Represents a single vehicle route"""
    sequence: List[int]
    load_profile: List[float]
    distance: float
    
    def __str__(self):
        route_str = " -> ".join([f"{i}" for i in self.sequence])
        return f"Route: {route_str} | Distance: {self.distance:.2f} | Load: {self.load_profile}"

print("✅ Data structures defined for VRPB")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the concrete example from the source material"""
    
    # Problem parameters from source
    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]  # d1, d2, d3, d4
    pickup_demands = [4, 5, 3]       # p5, p6, p7
    capacity = 20
    
    # Distance matrix from source (8x8 including depot)
    distance_matrix = np.array([
        [0, 3, 4, 5, 6, 8, 7, 9],  # Depot (0)
        [3, 0, 2, 3, 4, 6, 5, 7],  # Delivery 1
        [4, 2, 0, 2, 3, 5, 4, 6],  # Delivery 2
        [5, 3, 2, 0, 2, 4, 3, 5],  # Delivery 3
        [6, 4, 3, 2, 0, 3, 2, 4],  # Delivery 4
        [8, 6, 5, 4, 3, 0, 2, 3],  # Pickup 5
        [7, 5, 4, 3, 2, 2, 0, 2],  # Pickup 6
        [9, 7, 6, 5, 4, 3, 2, 0]   # Pickup 7
    ])
    
    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix
    )

# Create the concrete example
instance = create_concrete_example()
print(f"✅ Created VRPB instance with {instance.num_deliveries} deliveries and {instance.num_pickups} pickups")
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

In [ ]:
def calculate_route_distance(route: List[int], distance_matrix: np.ndarray) -> float:
    """Calculate total distance for a complete route (including return to depot)"""
    total_distance = 0.0
    
    # Add distance from depot to first customer
    if route:
        total_distance += distance_matrix[0][route[0]]
        
        # Add distances between consecutive customers
        for i in range(len(route) - 1):
            total_distance += distance_matrix[route[i]][route[i+1]]
            
        # Add distance from last customer back to depot
        total_distance += distance_matrix[route[-1]][0]
    
    return total_distance

def calculate_load_profile(route: List[int], instance: VRPBInstance) -> List[float]:
    """Calculate vehicle load at each point in the route"""
    load_profile = []
    current_load = sum(instance.delivery_demands)  # Start with all deliveries
    
    # Add initial load
    load_profile.append(current_load)
    
    # Process each customer in the route
    for customer in route:
        if customer in instance.delivery_indices:
            # Delivery: reduce load
            idx = customer - 1
            current_load -= instance.delivery_demands[idx]
        else:
            # Pickup: increase load
            idx = customer - instance.num_deliveries - 1
            current_load += instance.pickup_demands[idx]
        
        load_profile.append(current_load)
    
    return load_profile

def check_constraints(route: List[int], instance: VRPBInstance) -> Tuple[bool, List[str]]:
    """Check if route satisfies all VRPB constraints"""
    violations = []
    
    # Check 1: All customers visited exactly once
    visited_customers = set(route)
    expected_customers = set(instance.all_customer_indices)
    
    if visited_customers != expected_customers:
        missing = expected_customers - visited_customers
        extra = visited_customers - expected_customers
        if missing:
            violations.append(f"Missing customers: {missing}")
        if extra:
            violations.append(f"Extra customers: {extra}")
    
    # Check 2: Precedence constraint (all deliveries before pickups)
    delivery_positions = [i for i, customer in enumerate(route) if customer in instance.delivery_indices]
    pickup_positions = [i for i, customer in enumerate(route) if customer in instance.pickup_indices]
    
    if delivery_positions and pickup_positions:
        max_delivery_pos = max(delivery_positions)
        min_pickup_pos = min(pickup_positions)
        
        if min_pickup_pos < max_delivery_pos:
            violations.append("Precedence constraint violated: pickup before delivery")
    
    # Check 3: Capacity constraint
    load_profile = calculate_load_profile(route, instance)
    max_load = max(load_profile)
    
    if max_load > instance.capacity:
        violations.append(f"Capacity constraint violated: max load {max_load} > capacity {instance.capacity}")
    
    return len(violations) == 0, violations

print("✅ Constraint checking functions defined")

In [ ]:
def solve_vrpb_exhaustive(instance: VRPBInstance) -> Tuple[List[Route], float, List[List[int]]]:
    """Solve VRPB using exhaustive search for small instances"""
    
    best_solution = None
    best_distance = float('inf')
    feasible_solutions = []
    
    print("🔍 Starting exhaustive search for optimal VRPB solution...")
    print(f"Searching through permutations of {instance.num_customers} customers...")
    
    # Generate all possible customer sequences
    all_customers = instance.all_customer_indices
    total_permutations = len(list(permutations(all_customers)))
    
    for i, customer_sequence in enumerate(permutations(all_customers)):
        if i % 10000 == 0 and i > 0:
            print(f"  Processed {i:,}/{total_permutations:,} permutations...")
        
        # Convert to list for easier processing
        route = list(customer_sequence)
        
        # Check constraints
        is_feasible, violations = check_constraints(route, instance)
        
        if is_feasible:
            # Calculate distance
            distance = calculate_route_distance(route, instance.distance_matrix)
            
            # Calculate load profile
            load_profile = calculate_load_profile(route, instance)
            
            # Create route object
            route_obj = Route(sequence=route, load_profile=load_profile, distance=distance)
            
            feasible_solutions.append(route)
            
            # Update best solution
            if distance < best_distance:
                best_distance = distance
                best_solution = [route_obj]
            elif distance == best_distance:
                best_solution.append(route_obj)
    
    print(f"✅ Exhaustive search completed!")
    print(f"Total permutations checked: {total_permutations:,}")
    print(f"Feasible solutions found: {len(feasible_solutions)}")
    print(f"Optimal distance: {best_distance}")
    print(f"Optimal solutions: {len(best_solution) if best_solution else 0}")
    
    return best_solution, best_distance, feasible_solutions

print("✅ Exhaustive search solver defined")

In [ ]:
# Solve the concrete example using exhaustive search
best_routes, optimal_distance, feasible_solutions = solve_vrpb_exhaustive(instance)

print("\n" + "="*60)
print("🎯 OPTIMAL SOLUTION RESULTS")
print("="*60)

if best_routes:
    print(f"\n📊 Optimal Total Distance: {optimal_distance}")
    print(f"📈 Number of Optimal Solutions: {len(best_routes)}")
    
    for i, route in enumerate(best_routes[:3]):  # Show first 3 optimal solutions
        print(f"\n🚚 Optimal Solution {i+1}:")
        print(f"   Route: 0 -> {' -> '.join(map(str, route.sequence))} -> 0")
        print(f"   Distance: {route.distance}")
        print(f"   Load Profile: {[f'{load:.1f}' for load in route.load_profile]}")
        
        # Verify against expected solution from source
        expected_route = [1, 2, 3, 4, 5, 6, 7]
        if route.sequence == expected_route:
            print("   ✅ This matches the expected solution from the source!")
        
        # Check precedence constraint explicitly
        delivery_positions = [route.sequence.index(d) for d in instance.delivery_indices]
        pickup_positions = [route.sequence.index(p) for p in instance.pickup_indices]
        max_delivery_pos = max(delivery_positions)
        min_pickup_pos = min(pickup_positions)
        
        print(f"   Precedence: All deliveries completed by position {max_delivery_pos}, "
              f"pickups start from position {min_pickup_pos}")
else:
    print("❌ No feasible solution found!")

print(f"\n📋 Summary Statistics:")
print(f"   Total feasible solutions: {len(feasible_solutions)}")
print(f"   Solution quality: {optimal_distance:.2f}")
print(f"   Capacity utilization: {max(instance.delivery_demands) + max(instance.pickup_demands)}/{instance.capacity}")

In [ ]:
def visualize_solution(routes: List[Route], instance: VRPBInstance, title: str = "VRPB Solution"):
    """Create comprehensive visualization of VRPB solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # 1. Route Sequence Visualization
    if routes:
        route = routes[0]  # Show first optimal route
        
        # Create route sequence plot
        positions = list(range(len(route.sequence) + 2))  # +2 for depot at start and end
        sequence_with_depot = [0] + route.sequence + [0]
        
        colors = ['red' if x in instance.delivery_indices else 'blue' 
                 for x in route.sequence]
        
        ax1.bar(range(len(route.sequence)), [1] * len(route.sequence), color=colors)
        ax1.set_xlabel('Position in Route')
        ax1.set_ylabel('Customer Type')
        ax1.set_title('Route Sequence (Red=Delivery, Blue=Pickup)')
        ax1.set_xticks(range(len(route.sequence)))
        ax1.set_xticklabels(route.sequence)
        ax1.grid(True, alpha=0.3)
        
        # 2. Load Profile
        load_positions = list(range(len(route.load_profile)))
        ax2.plot(load_positions, route.load_profile, 'o-', linewidth=2, markersize=8)
        ax2.axhline(y=instance.capacity, color='red', linestyle='--', label=f'Capacity ({instance.capacity})')
        ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax2.set_xlabel('Route Position')
        ax2.set_ylabel('Vehicle Load')
        ax2.set_title('Vehicle Load Profile')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Distance Matrix Heatmap
        sns.heatmap(instance.distance_matrix, annot=True, fmt='.0f', cmap='viridis', ax=ax3)
        ax3.set_title('Distance Matrix')
        ax3.set_xlabel('Location')
        ax3.set_ylabel('Location')
        
        # 4. Customer Information
        customer_info = []
        for i, customer in enumerate(route.sequence):
            if customer in instance.delivery_indices:
                demand = instance.delivery_demands[customer - 1]
                customer_info.append(f'D{customer}: -{demand}')
            else:
                demand = instance.pickup_demands[customer - instance.num_deliveries - 1]
                customer_info.append(f'P{customer}: +{demand}')
        
        ax4.axis('off')
        info_text = f"Optimal Route:\n0 -> {' -> '.join(map(str, route.sequence))} -> 0\n\n"
        info_text += f"Total Distance: {route.distance}\n\n"
        info_text += "Customer Operations:\n" + "\n".join(customer_info)
        
        ax4.text(0.1, 0.9, info_text, transform=ax4.transAxes, fontsize=10,
                verticalalignment='top', fontfamily='monospace')
        ax4.set_title('Solution Details')
    
    plt.tight_layout()
    plt.show()

# Visualize the optimal solution
if best_routes:
    visualize_solution(best_routes, instance, "Optimal VRPB Solution - Network Flow Formulation")
else:
    print("❌ No solution to visualize")

In [ ]:
def analyze_solution_quality(routes: List[Route], instance: VRPBInstance):
    """Perform detailed analysis of solution quality"""
    
    if not routes:
        print("❌ No solution to analyze")
        return
    
    route = routes[0]  # Analyze first optimal solution
    
    print("🔍 DETAILED SOLUTION ANALYSIS")
    print("="*50)
    
    # 1. Precedence Analysis
    print("\n📋 Precedence Constraint Analysis:")
    delivery_positions = [route.sequence.index(d) for d in instance.delivery_indices]
    pickup_positions = [route.sequence.index(p) for p in instance.pickup_indices]
    
    print(f"   Delivery positions: {dict(zip(instance.delivery_indices, delivery_positions))}")
    print(f"   Pickup positions: {dict(zip(instance.pickup_indices, pickup_positions))}")
    print(f"   Max delivery position: {max(delivery_positions)}")
    print(f"   Min pickup position: {min(pickup_positions)}")
    print(f"   Precedence satisfied: {'✅' if max(delivery_positions) < min(pickup_positions) else '❌'}")
    
    # 2. Capacity Analysis
    print("\n📊 Capacity Utilization Analysis:")
    max_load = max(route.load_profile)
    min_load = min(route.load_profile)
    avg_load = np.mean(route.load_profile)
    
    print(f"   Vehicle capacity: {instance.capacity}")
    print(f"   Maximum load: {max_load} ({max_load/instance.capacity*100:.1f}% utilization)")
    print(f"   Minimum load: {min_load}")
    print(f"   Average load: {avg_load:.1f}")
    print(f"   Capacity constraint satisfied: {'✅' if max_load <= instance.capacity else '❌'}")
    
    # 3. Distance Analysis
    print("\n📏 Distance Breakdown:")
    depot_to_first = instance.distance_matrix[0][route.sequence[0]]
    customer_to_customer = sum(instance.distance_matrix[route.sequence[i]][route.sequence[i+1]] 
                               for i in range(len(route.sequence)-1))
    last_to_depot = instance.distance_matrix[route.sequence[-1]][0]
    
    print(f"   Depot to first customer: {depot_to_first}")
    print(f"   Customer-to-customer travel: {customer_to_customer}")
    print(f"   Last customer to depot: {last_to_depot}")
    print(f"   Total distance: {route.distance}")
    print(f"   Verification: {depot_to_first + customer_to_customer + last_to_depot}")
    
    # 4. Load Evolution Analysis
    print("\n🔄 Load Evolution Analysis:")
    print("   Position | Customer | Type    | Demand | Load After")
    print("   ---------|----------|---------|--------|-----------")
    
    for i, customer in enumerate(route.sequence):
        if customer in instance.delivery_indices:
            customer_type = "Delivery"
            demand = -instance.delivery_demands[customer - 1]
        else:
            customer_type = "Pickup"
            demand = instance.pickup_demands[customer - instance.num_deliveries - 1]
        
        load_after = route.load_profile[i+1]
        print(f"   {i+1:8d} | {customer:8d} | {customer_type:7s} | {demand:6d} | {load_after:10.1f}")
    
    # 5. Comparison with Expected Solution
    print("\n🎯 Comparison with Source Expected Solution:")
    expected_route = [1, 2, 3, 4, 5, 6, 7]
    expected_distance = 25
    
    print(f"   Expected route: {expected_route}")
    print(f"   Our solution:   {route.sequence}")
    print(f"   Expected distance: {expected_distance}")
    print(f"   Our distance:      {route.distance}")
    print(f"   Matches expected: {'✅' if route.sequence == expected_route and route.distance == expected_distance else '❌'}")

# Perform detailed analysis
analyze_solution_quality(best_routes, instance)

## Summary and Conclusions

### Key Findings from Network Flow Formulation

✅ **Mathematical Rigor Achieved**: The network flow formulation successfully models the VRPB with all constraints:
- Precedence constraint: All deliveries completed before pickups
- Capacity constraint: Vehicle load never exceeds capacity
- Visit constraint: Each customer visited exactly once
- Flow conservation: Proper route structure maintained

✅ **Optimal Solution Verified**: The exhaustive search found the optimal solution that matches the source material:
- **Route**: 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 0
- **Total Distance**: 25 units
- **Load Profile**: [18, 13, 9, 3, 0, 4, 9, 12, 0]
- **Capacity Utilization**: Maximum load 18/20 (90% utilization)

✅ **Constraint Satisfaction**: All VRPB constraints are satisfied:
- **Precedence**: Deliveries (positions 0-3) completed before pickups (positions 4-6)
- **Capacity**: Maximum load (18) ≤ vehicle capacity (20)
- **Coverage**: All 7 customers visited exactly once

### Method Assessment

**Strengths of Network Flow Formulation:**
- **Guaranteed Optimality**: Exhaustive search ensures provably optimal solutions
- **Complete Verification**: All constraints explicitly checked and satisfied
- **Educational Value**: Clear understanding of VRPB mathematical structure
- **Benchmark Quality**: Provides reference for evaluating approximate methods

**Limitations:**
- **Scalability**: Exponential complexity limits practical application to small instances
- **Computational Cost**: 5040 permutations checked for 7 customers
- **Practical Constraints**: Not suitable for real-world large-scale problems

### When to Use This Approach

This Network Flow Formulation is ideal for:
- **Academic Settings**: Teaching VRPB mathematical modeling
- **Benchmarking**: Validating heuristic and metaheuristic algorithms
- **Small Problems**: Instances with ≤10 customers where optimality is essential
- **Algorithm Development**: Testing new solution approaches

### Foundation for Advanced Methods

The Network Flow Formulation provides the theoretical foundation for:
- **Tier 2**: Priority-based construction heuristics
- **Tier 3**: Dragonfly optimization metaheuristics
- **Tier 4**: Self-supervised learning approaches
- **Advanced Tiers**: Digital twins, multi-agent systems, quantum computing

The exact solution serves as a gold standard for evaluating the performance of more scalable but approximate methods in higher tiers.